In [1]:
import pennylane as qml
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from itertools import product
import yfinance as yf

In [2]:
n_assets=3
n_bits=2     
N=n_assets*n_bits 

def build_qubo_matrix(mean_returns,cov_matrix,lambda_risk,lambda_budget,lambda_sector,sector_map,sector_cap):
    Q = np.zeros((N,N))

    for i in range(n_assets):
        for k in range(n_bits):
            idx = i * n_bits + k
            weight_contribution = (2**k) /(2**n_bits -1)
            Q[idx,idx] -= mean_returns[i] * weight_contribution

    for i in range(n_assets):
        for j in range(n_assets):
            for k in range(n_bits):
                for m in range(n_bits):
                    row = i * n_bits + k
                    col = j * n_bits + m
                    wi = (2**k) /(2**n_bits -1)
                    wj = (2**m)/(2**n_bits -1)
                    Q[row,col] += lambda_risk * cov_matrix[i,j] * wi * wj

    for i in range(n_assets):
        for k in range(n_bits):
            idx= i *n_bits + k
            wi = (2**k)/ (2**n_bits-1)
            Q[idx,idx] += lambda_budget * wi * (wi-2) 

            for j in range(n_assets):
                for m in range(n_bits):
                    col = j*n_bits + m
                    if col> idx:
                        wj = (2**m)/(2**n_bits -1)
                        Q[idx,col] += 2 * lambda_budget * wi * wj

    sectors = set(sector_map.values())
    for sector in sectors:
        sector_assets = [i for i, s in sector_map.items() if s == sector]
        for i in sector_assets:
            for k in range(n_bits):
                idx = i*n_bits +k
                wi = (2**k)/(2**n_bits -1)
                Q[idx,idx] += lambda_sector * wi * (wi -2 * sector_cap)
                for j in sector_assets:
                    for m in range(n_bits):
                        col= j*n_bits + m
                        if col>idx:
                            wj = (2**m)/(2**n_bits-1)
                            Q[idx,col]+= 2 * lambda_sector * wi * wj
    return Q


tickers = ["AAPL", "MSFT","JPM"]
data= yf.download(tickers, start = "2020-01-01", end = "2026-01-01")
prices=data["Close"]
log_returns=np.log(prices/prices.shift(1)).dropna()

mean_returns = log_returns.mean().values * 252
cov_matrix = log_returns.cov().values * 252

# scetor map: 0=tech, 1=finance. ticker:sector
sector_map = {0:0,1:0,2:1}

Q= build_qubo_matrix(mean_returns=mean_returns,cov_matrix=cov_matrix,lambda_risk=1.0,lambda_budget=5.0
                    ,lambda_sector=2.0,sector_map=sector_map,sector_cap=0.6)
print(f"Q matrix shape: {Q.shape}")


[*********************100%***********************]  3 of 3 completed

Q matrix shape: (6, 6)


In [7]:
def qubo_to_ising_hamiltonian(Q):
    """
    Substitution: x_i=(1-s_i)/2
    Returns:
        coeffs: list of Hamiltonian term coefficients 
        obs: list of PennyLane Pauli_Z observables
    """
    n=Q.shape[0]
    coeffs=[]
    obs=[]

    offset=0.0
    linear=np.zeros(n)
    quadratic={}

    for i in range(n):
        for j in range(n):
            if Q[i,j] == 0:
                continue
            if i==j:
                offset += Q[i,i]/2
                linear[i] += -Q[i,i]/2
            else:
                offset += Q[i,j]/4
                linear[i] += -Q[i,j]/4
                linear[j] += -Q[i,j]/4
                key=tuple(sorted((i,j)))
                quadratic[key]=quadratic.get(key,0) + Q[i,j]/4

    for i in range(n):
        if linear[i] != 0:
            coeffs.append(linear[i])
            obs.append(qml.PauliZ(i))
    for (i,j), coeff in quadratic.items():
        if coeff != 0:
            coeffs.append(coeff)
            obs.append(qml.PauliZ(i) @ qml.PauliZ(j))

    return coeffs, obs, offset

coeffs,obs,offset = qubo_to_ising_hamiltonian(Q)
cost_hamiltonian = qml.Hamiltonian(coeffs,obs)

print(f"No of Hamiltonian terms: {len(coeffs)}")
print(f"Constant offset: {offset:.4f}")

No of Hamiltonian terms: 21
Constant offset: -2.0377


In [4]:
n_qubits=N
dev = qml.device("default.qubit",wires=n_qubits)

def cost_layer(gamma,cost_hamiltonian):
    #applies e^(-i*gamma*H_cost)
    qml.templates.ApproxTimeEvolution(cost_hamiltonian,gamma,1)

def mixer_layer(beta):
    for i in range(n_qubits):
        qml.RX(2*beta,wires=i)

@qml.qnode(dev)
def qaoa_circuit(params,p):
    gammas = params[:p]
    betas = params[p:]

    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    for layer in range(p):
        cost_layer(gammas[layer],cost_hamiltonian)
        mixer_layer(betas[layer])

    return qml.expval(cost_hamiltonian)
    

In [20]:
np.random.seed(42)
def run_qaoa(p=1,n_restarts=5,maxiter=300):
    #runs qaoa with p layers
    best_result=None
    best_cost=np.inf
    history=[]

    for restart in range(n_restarts):
        params0 = np.random.uniform(0,np.pi,size=2*p)
        result = minimize(lambda params: qaoa_circuit(params,p), params0,method="COBYLA",options={'maxiter':maxiter})
        history.append(result.fun)
        if result.fun<best_cost:
            best_cost=result.fun
            best_result=result

    return best_result, best_cost,history

result_p1,cost_p1,history_p1 = run_qaoa(p=1,n_restarts=5)
print(f"QAOA (p=1) best expectation value: {cost_p1:.4f}")
print(f"Energy for p=1(add offset back): {cost_p1 + offset: .4f}")
result_p2,cost_p2,history_p2 = run_qaoa(p=2,n_restarts=15)
print(f"QAOA (p=2) best expectation value: {cost_p2:.4f}")
print(f"Energy for p=2 (add offset back): {cost_p2 + offset: .4f}")

QAOA (p=1) best expectation value: -3.0325
Energy for p=1(add offset back): -5.0702
QAOA (p=2) best expectation value: -3.1606
Energy for p=2 (add offset back): -5.1983


In [18]:
def brute_force_qubo(Q):
    N = Q.shape[0]
    best_energy = np.inf
    best_x = None

    for bits in product([0,1], repeat=N):
        x=np.array(bits)
        energy = x@Q@x
        if energy < best_energy:
            best_energy = energy
            best_x = x.copy
    return best_x, best_energy

x_brute,energy_brute = brute_force_qubo(Q)
print(f"Brute force optimal energy: {energy_brute: .5f}")


Brute force optimal energy: -6.41736


In [24]:
@qml.qnode(dev)
def sample_circuit(params,p,shots=1000):
    gammas = params[:p]
    betas = params[p:]

    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    for layer in range(p):
        cost_layer(gammas[layer],cost_hamiltonian)
        mixer_layer(betas[layer])

    return qml.sample(wires=range(n_qubits))

dev_sampling = qml.device("default.qubit",wires=n_qubits,shots=1000)
sample_circuit = qml.QNode(sample_circuit.func,dev_sampling)
sample_circuit = qml.set_shots(sample_circuit,shots=1000)

best_params = result_p2.x if cost_p2<cost_p1 else result_p1.x
best_p = 2 if cost_p2<cost_p1 else 1

samples = sample_circuit(best_params,best_p)

from collections import Counter
sample_tuples = [tuple(s) for s in samples]
most_common = Counter(sample_tuples).most_common(5)

print("Top 5 most frequent measured solutions:")
for bitstring, count in most_common:
    x=np.array(bitstring)
    energy= x@Q@x
    print(f"{bitstring} | count: {count:4d}/1000 | QUBO energy: {energy:.4f}")

Top 5 most frequent measured solutions:
(np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0)) | count:   52/1000 | QUBO energy: -6.4174
(np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0)) | count:   48/1000 | QUBO energy: -5.5204
(np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0)) | count:   47/1000 | QUBO energy: -6.3928
(np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0)) | count:   47/1000 | QUBO energy: -6.4165
(np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0)) | count:   44/1000 | QUBO energy: -4.2208
